In [0]:

# ==========================================
# NOTEBOOK 3: GOLD ANALYTICS
# Retail Sales Data Engineering Pipeline
# ==========================================

# 1. IMPORT LIBRARIES

from pyspark.sql.functions import (
    col, sum, countDistinct
)
from pyspark.sql.types import DecimalType


# 2. READ SILVER DELTA TABLE

silver_table = spark.table(
    "retail_practice.default.silver_retail_sales"
)

print("Silver table loaded successfully")
print("Silver records:", silver_table.count())


# 3. MONTHLY SALES ANALYSIS

gold_monthly = silver_table.filter(
    col("sales_category") == "Completed"
).groupBy(
    "order_year",
    "order_month",
    "Month_name"
).agg(
    sum("Net_sale").cast(
        DecimalType(18, 2)
    ).alias("monthly_revenue")
).orderBy(
    "order_year",
    "order_month"
)


# 4. YEARLY SALES ANALYSIS

gold_yearly = silver_table.filter(
    col("sales_category") == "Completed"
).groupBy(
    "order_year"
).agg(
    sum("Net_sale").cast(
        DecimalType(18, 2)
    ).alias("yearly_revenue"),
    countDistinct("order_id").alias("total_orders"),
    sum("quantity").alias("total_units_sold")
).orderBy("order_year")


# 5. CATEGORY-WISE SALES ANALYSIS

gold_category = silver_table.filter(
    col("sales_category") == "Completed"
).groupBy(
    "category"
).agg(
    sum("Net_sale").cast(
        DecimalType(18, 2)
    ).alias("total_revenue"),
    sum("quantity").alias("total_units_sold")
).orderBy(
    col("total_revenue").desc()
)


# 6. PRODUCT-WISE SALES ANALYSIS

gold_product = silver_table.filter(
    col("sales_category") == "Completed"
).groupBy(
    "product_id",
    "product_name",
    "category"
).agg(
    sum("Net_sale").cast(
        DecimalType(18, 2)
    ).alias("total_revenue"),
    sum("quantity").alias("total_units_sold")
).orderBy(
    col("total_revenue").desc()
)

print("Gold aggregations completed")


# 7. SAVE GOLD DELTA TABLES

gold_monthly.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "retail_practice.default.gold_monthly_sales"
    )

gold_yearly.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "retail_practice.default.gold_yearly_sales"
    )

gold_category.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "retail_practice.default.gold_category_sales"
    )

gold_product.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "retail_practice.default.gold_product_sales"
    )

print("All Gold Delta tables saved successfully")


# 8. VALIDATE GOLD TABLES

gold_tables = [
    "gold_monthly_sales",
    "gold_yearly_sales",
    "gold_category_sales",
    "gold_product_sales"
]

print("----------------------------------")
print("GOLD PIPELINE VALIDATION")
print("----------------------------------")

for table_name in gold_tables:

    df = spark.table(
        f"retail_practice.default.{table_name}"
    )

    print(table_name, ":", df.count(), "rows")

print("----------------------------------")


# 9. DISPLAY GOLD ANALYTICS

print("MONTHLY SALES")
display(spark.table(
    "retail_practice.default.gold_monthly_sales"
))

print("YEARLY SALES")
display(spark.table(
    "retail_practice.default.gold_yearly_sales"
))

print("CATEGORY SALES")
display(spark.table(
    "retail_practice.default.gold_category_sales"
))

print("TOP 10 PRODUCTS")
display(spark.table(
    "retail_practice.default.gold_product_sales"
).limit(10))
